# clip-grad-norm-pre-step — faded example 3: Apply the in-place rescale factor when over threshold

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`. The last cell reports your progress on the `Optimizer: clip_grad_norm pre-step` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: clip_grad_norm pre-step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`clip-grad-norm-pre-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "clip-grad-norm-pre-step"
DD_SUBTOPIC = "Optimizer: clip_grad_norm pre-step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the global norm exceeds `max_norm`, every gradient is multiplied in place by `max_norm / (total + 1e-6)`, matching PyTorch's reference (the epsilon avoids divide-by-zero). This must mutate the original `.grad` tensors so the optimizer reads the clipped values by reference.

## Faded exercise 3

### Faded - apply the rescale

The global norm `total` is already computed and the over-threshold branch is entered. Fill in the in-place multiply that rescales each gradient `g` so the resulting global norm equals `max_norm`. Use PyTorch's `+ 1e-6` convention in the denominator.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
t.manual_seed(0)

def manual_clip(params, max_norm):
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        return 0.0
    total = sum((g.detach() ** 2).sum() for g in grads).sqrt()
    if total.item() > max_norm:
        scale = max_norm / (total + 1e-6)
        for g in grads:
            g.mul_(scale)
    return total.item()

p = t.zeros(4, requires_grad=True)
p.grad = t.full((4,), 4.0)
pre = manual_clip([p], max_norm=2.0)
print("pre-clip norm:", round(pre, 4))
print("post-clip norm:", round(p.grad.norm().item(), 4))


def _test():
    import torch.nn.utils as nn_utils
    p = t.zeros(4, requires_grad=True)
    p.grad = t.full((4,), 4.0)
    pre = manual_clip([p], max_norm=2.0)
    # pre-clip norm sqrt(4*16) = 8.0
    assert abs(pre - 8.0) < 1e-4, f"expected pre-clip 8.0, got {pre}"
    # torch reference for post-clip grad
    pr = t.zeros(4, requires_grad=True)
    pr.grad = t.full((4,), 4.0)
    nn_utils.clip_grad_norm_([pr], max_norm=2.0)
    assert t.allclose(p.grad, pr.grad, atol=1e-4), "post-clip grad mismatch vs torch reference"
    # post-clip norm should be ~2.0
    assert abs(p.grad.norm().item() - 2.0) < 1e-3, "post-clip norm should equal max_norm"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
t.manual_seed(0)

def manual_clip(params, max_norm):
    grads = [p.grad for p in params if p.grad is not None]
    if not grads:
        return 0.0
    total = sum((g.detach() ** 2).sum() for g in grads).sqrt()
    if total.item() > max_norm:
        scale = max_norm / (total + 1e-6)
        for g in grads:
            g.mul_(scale)
    return total.item()

p = t.zeros(4, requires_grad=True)
p.grad = t.full((4,), 4.0)
pre = manual_clip([p], max_norm=2.0)
print("pre-clip norm:", round(pre, 4))
print("post-clip norm:", round(p.grad.norm().item(), 4))
```
</details>